AUGMENTACJA OFFLINE I OVERSAMPLING

In [ ]:
!nvidia-smi -L
%pip install -q "ultralytics>=8.3,<9" codecarbon albumentations "pandas>=2,<4"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
![ -d /content/garbage-yolo ] || git clone -q --depth 1 https://github.com/PovMiki/garbage-yolo /content/garbage-yolo

In [ ]:
import os, time, yaml, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import albumentations as A
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops
from codecarbon import EmissionsTracker

DS = Path("/content/garbage-yolo/dataset_clean")
AUG = Path("/content/aug")
OUT = Path("/content/drive/MyDrive/garbage_yolo_aug")
BASELINE = OUT / "baseline"
RUNS = OUT / "runs"
RESULTS = OUT / "results_comparison.csv"
PER_CLASS = OUT / "results_per_class.csv"
EPOCHS, BATCH, IMGSZ, WORKERS, SEED = 50, 16, 640, 2, 42
ROTATE_90 = True
TARGETS = {"PLASTIC": 4000, "PAPER": 3500, "CARDBOARD": 3500, "METAL": 3500} #docelowa ilosc o ile mozliwe
MAX_COPIES = 2                                                               #maksymalna ilosc kopii zdj zeby nie bylo przeuczenia

NO_AUG = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0, scale=0.0, shear=0.0,
              perspective=0.0, flipud=0.0, fliplr=0.0, bgr=0.0, mosaic=0.0, mixup=0.0,
              copy_paste=0.0, erasing=0.0)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
assert DEVICE == 0, "Brak GPU"
assert (BASELINE / "best.pt").is_file(), f"Brak {BASELINE / 'best.pt'}"

cfg = yaml.safe_load((DS / "data.yaml").read_text())
cfg["path"] = str(DS)
NAMES = dict(enumerate(cfg["names"])) if isinstance(cfg["names"], list) else {int(k): v for k, v in cfg["names"].items()} #numer -> nazwa
DATA = DS / "data_colab.yaml"
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
TEST_IMAGES = sorted((DS / "test/images").glob("*"))
RUNS.mkdir(parents=True, exist_ok=True)

Wybor zdjec do oversamplingu

In [ ]:
def plan_copies(df):
    names = sorted({c for classes in df["classes"] for c in classes})
    boxes = np.array([[classes.count(c) for c in names] for classes in df["classes"]]) #wiersz = zdjecie, kolumna = klasa, wartosc = liczba obiektow
    count = boxes.sum(axis=0)                                                          #ilosc obiektow kazdej klasy
    target = np.array([TARGETS.get(c, 0) for c in names])                              #klasy spoza targets (brak wzmacniania)
    bio = names.index("BIODEGRADABLE")
    copies = np.zeros(len(df), dtype=int)
    stuck = set() #klasy niemozliwe do podniesienia - brak zdj
    while True:
        under = [j for j in range(len(names)) if j not in stuck and count[j] < target[j]]
        if not under: #wszystkie klasy osiagnely cel
            break
        rarest = min(under, key=lambda j: count[j] / target[j])           #klasa najdalej od celu procentowo
        useful = np.where(count < target, 1, -1)                          #1 brakuje -1 juz osiagnela cel
        useful[target == 0] = 0                                           #GLASS nie pomaga
        useful[bio] = -1                                                  #biodegradable za duzo
        score = boxes @ useful                                            #obiekty potrzebne - obiekty zbedne
        ok = (boxes[:, rarest] > 0) & (copies < MAX_COPIES) & (score > 0) #jest brakujaca klasa, ma < 2 kopii, wiecej pomaga niz szkodzi
        if not ok.any():
            stuck.add(rarest)
            continue
        best = np.argmax(np.where(ok, boxes[:, rarest] * 1000 + score - copies * 10**6, -10**9))
        copies[best] += 1
        count += boxes[best]
    return copies
"""
kolejnosc najlepszego zdj:
- copies * 10**6            - najpierw zdjecia jeszcze niekopiowane zeby sie nie przeuczal
boxes[:, rarest] * 1000     - najwiecej obiektow brakujacej klasy
score                       - przy remisie mniej zbednych obiektow
-10**9                      - zdjecia ktorych nie wolno wybrac nigdy nie wygraja
duze mnozniki zeby wazniejszy warunek zawsze wygrywal
"""


rows = []
for img in sorted((DS / "train/images").iterdir()):
    label = DS / "train/labels" / f"{img.stem}.txt"
    classes = [NAMES[int(c)] for c in np.loadtxt(label, ndmin=2)[:, 0]] #ndmin=2 do tabeli nawet przy 1 ramce, kolumna 0 = klasa
    rows.append({"image": img, "label": label, "classes": classes})
train_df = pd.DataFrame(rows)
train_df["strong"] = plan_copies(train_df) #ilosc kopii na zdj 0,1,2

boxes = train_df.explode("classes") #jeden wiersz = jedna ramka
original = boxes["classes"].value_counts()
balanced = original + boxes.groupby("classes")["strong"].sum() #kazda kopia dodaje wszystkie ramki ze zdj
versions = pd.DataFrame({"original": original, "aug_balanced": balanced}).sort_values("original", ascending=False)
table = pd.concat({name: pd.DataFrame({"liczba": v, "%": (v / v.sum() * 100).round(1)})
                   for name, v in versions.items()}, axis=1)
display(table)
display(train_df["strong"].value_counts().sort_index().rename("zdjec").rename_axis("kopie"))

Generowanie zbioru

In [ ]:
BBOX = A.BboxParams(format="yolo",
                    label_fields=["classes"], #klasa przypisana do ramki - usuwana razem z nia
                    clip=True)                #ramka wystaje poza kadr -> przycinanie jej do brzegu

ROTATIONS = [[A.Transpose(p=1), A.HorizontalFlip(p=1)], [A.HorizontalFlip(p=1), A.VerticalFlip(p=1)]] #90 i 180 stopni, bez czarnych rogow


def strong_copy(k):
    rotate = ROTATIONS[k % len(ROTATIONS)] if ROTATE_90 else [] #kopia 1 -> 90, kopia 2 -> 180, zeby kopie sie roznily
    return A.Compose(rotate + [
        A.HorizontalFlip(p=0.5),                                                           #p = szansa ze krok sie wykona
        A.RandomSizedBBoxSafeCrop(height=IMGSZ, width=IMGSZ, p=0.5),                       #przyblizenie bez ucinania ramek
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.9),       #inne oswietlenie
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=30, val_shift_limit=0, p=0.5), #barwa tylko +-5 zeby nie zmienic koloru materialu
    ], bbox_params=BBOX, seed=SEED + 1 + k) #inny seed dla kazdej kopii


STRONG = [strong_copy(k) for k in range(MAX_COPIES)]


def save(image, bboxes, classes, image_path):
    if len(bboxes) == 0: #zdjecie bez ramek nic nie uczy - pomijamy
        return
    cv2.imwrite(str(image_path), image)
    label_path = image_path.parent.parent / "labels" / f"{image_path.stem}.txt" #yolo szuka etykiet w labels obok images
    label_path.write_text("".join(f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"
                                  for c, (x, y, w, h) in zip(classes, bboxes)))


def augment(row, transform, tag, root):
    image = cv2.imread(str(row.image))
    label = np.loadtxt(row.label, ndmin=2)
    out = transform(image=image, bboxes=label[:, 1:], classes=label[:, 0])
    save(out["image"], out["bboxes"], out["classes"], root / "images" / f"{row.image.stem}_{tag}.jpg")


def write_yaml(root):
    data = {"path": str(root), "train": "images", "val": str(DS / "valid/images"), #valid i test bez zmian - oryginalne zdjecia
            "test": str(DS / "test/images"), "names": NAMES}
    (root / "data.yaml").write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return root / "data.yaml"


def make_balanced():
    root = AUG / "aug_balanced"
    if (root / "data.yaml").is_file(): #zbior juz wygenerowany w tej sesji
        return root / "data.yaml"
    (root / "images").mkdir(parents=True, exist_ok=True)
    (root / "labels").mkdir(parents=True, exist_ok=True)
    for row in train_df.itertuples():
        os.symlink(row.image, root / "images" / row.image.name) #oryginal jako skrot zamiast kopii - oszczednosc miejsca
        shutil.copy(row.label, root / "labels" / row.label.name)
        for k in range(row.strong):
            augment(row, STRONG[k], f"strong{k}", root)
    return write_yaml(root)


start = time.perf_counter()
DATA_BALANCED = make_balanced()
print(f"czas : {(time.perf_counter() - start) / 60:.1f} min")
print(f"aug_balanced : {len(list((AUG / 'aug_balanced/images').iterdir()))} zdjec")

In [ ]:
def draw(image_path):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    label = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
    for c, x, y, bw, bh in np.loadtxt(label, ndmin=2):
        cv2.rectangle(image, (int((x - bw / 2) * w), int((y - bh / 2) * h)),
                      (int((x + bw / 2) * w), int((y + bh / 2) * h)), (0, 255, 0), 2)
    return image[:, :, ::-1] #cv2 BGR -> RGB dla matplotlib


examples = train_df[train_df["strong"] >= 2].sample(3, random_state=SEED) #zdj z 2 kopiami zeby pokazac obie
root = AUG / "aug_balanced/images"
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for r, row in enumerate(examples.itertuples()):
    for c, name in enumerate([row.image.name, f"{row.image.stem}_strong0.jpg", f"{row.image.stem}_strong1.jpg"]):
        if (root / name).exists():
            axes[r, c].imshow(draw(root / name))
        axes[r, c].set_title(["oryginal", "kopia 1", "kopia 2"][c])
        axes[r, c].axis("off")
plt.tight_layout()
plt.savefig(OUT / "augmentation_examples.png", dpi=120)
plt.show()

Trening

In [ ]:
def train(name, data):
    run = RUNS / name
    if (run / "done").exists(): #trening skonczony wczesniej - pomijamy
        return run
    run.mkdir(parents=True, exist_ok=True)
    #mierzy energie i CO2 treningu
    tracker = EmissionsTracker(project_name=name, output_dir=str(run), output_file="emissions.csv",
                               measure_power_secs=5, save_to_file=True, log_level="error")
    tracker.start()
    try:
        resume = (run / "weights/last.pt").exists() #colab sie rozlaczyl -> dokanczamy od ostatniej epoki
        model = YOLO(str(run / "weights/last.pt") if resume else "yolo11n.pt")
        model.add_callback("on_fit_epoch_end", lambda trainer: tracker.flush()) #zapis energii po kazdej epoce, zeby nie przepadla po rozlaczeniu
        if resume:
            model.train(resume=True)
        else:
            model.train(data=str(data), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                        workers=WORKERS, project=str(RUNS), name=name, exist_ok=True, **NO_AUG)
    finally:
        tracker.stop()
    (run / "done").touch() #znacznik ze trening sie skonczyl
    return run

Ewaluacja

In [ ]:
#cpu bo telefon nie ma mocnego gpu, po 1 zdj jak w aplikacji
def cpu_latency(model, images, repeats=3):
    model.predict(str(images[0]), imgsz=IMGSZ, device="cpu", verbose=False)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for img in images:
            model.predict(str(img), imgsz=IMGSZ, device="cpu", verbose=False)
        times.append((time.perf_counter() - start) / len(images) * 1000)
    return sum(times) / len(times), pd.Series(times).std()


def train_seconds(run):
    file = run / "results.csv"
    if not file.is_file():
        return 0
    t = pd.read_csv(file)["time"]
    return t.groupby((t.diff() < 0).cumsum()).max().sum() #po wznowieniu czas liczy sie od 0


def emissions(run):
    file = run / "emissions.csv"
    if not file.is_file():
        return {"train_min": None, "train_kWh": None, "train_CO2_kg": None}
    e = pd.read_csv(file).groupby("run_id")[["duration", "energy_consumed", "emissions"]].max().sum() #kazde uruchomienie osobno, potem suma
    seconds = max(e["duration"], train_seconds(run))
    scale = seconds / e["duration"] #dolicza energie z czasu ktory tracker zgubil przy rozlaczeniu
    return {"train_min": seconds / 60, "train_kWh": e["energy_consumed"] * scale,
            "train_CO2_kg": e["emissions"] * scale}


def evaluate(weights, name, run):
    model = YOLO(str(weights))
    #test - zdjecia ktorych model nie widzial przy treningu ani wyborze epoki
    m = model.val(data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
                  project=str(run), name="test", exist_ok=True, plots=True, verbose=False)
    p, r = float(m.box.mp), float(m.box.mr)
    ms, ms_std = cpu_latency(model, TEST_IMAGES[:100])
    row = {"model": name, "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
           "precision": p, "recall": r, "F1": 2 * p * r / (p + r + 1e-9),
           "ms_cpu": ms, "ms_cpu_std": ms_std, "FPS": 1000 / ms,
           "size_MB": os.path.getsize(weights) / 1e6,
           "GFLOPs": get_flops(model.model, IMGSZ),
           "params_M": sum(x.numel() for x in model.model.parameters()) / 1e6,
           "train_images": len(list((AUG / name / "images").iterdir())) if (AUG / name).exists() else len(train_df),
           **emissions(run)}
    per_class = pd.DataFrame({"model": name, "class": [m.names[int(c)] for c in m.box.ap_class_index],
                              "mAP50-95": [float(m.box.maps[int(c)]) for c in m.box.ap_class_index]})
    return row, per_class


def add_result(weights, name, run):
    df = pd.read_csv(RESULTS) if RESULTS.is_file() else pd.DataFrame(columns=["model"])
    if name in df["model"].values: #model juz oceniony, odswiezanie energii
        for key, value in emissions(run).items():
            if value is not None:
                df.loc[df["model"] == name, key] = value
        df.to_csv(RESULTS, index=False)
        return
    row, per_class = evaluate(weights, name, run)
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(RESULTS, index=False)
    per_class.to_csv(PER_CLASS, mode="a", header=not PER_CLASS.is_file(), index=False)


add_result(BASELINE / "best.pt", "baseline", BASELINE)

run = train("aug_balanced", DATA_BALANCED)
add_result(run / "weights/best.pt", "aug_balanced", run)

pd.read_csv(RESULTS).round(4)

Porownanie z baseline

In [ ]:
df = pd.read_csv(RESULTS).set_index("model")
change = ((df - df.loc["baseline"]) / df.loc["baseline"] * 100).round(1) #o ile % lepiej gorzej niz baseline
display(df.round(4))
display(change[["mAP50", "mAP50-95", "precision", "recall", "F1", "train_min", "train_kWh"]])

per_class = pd.read_csv(PER_CLASS).pivot_table(index="class", columns="model", values="mAP50-95")[df.index]
per_class_change = per_class.sub(per_class["baseline"], axis=0) #roznica kazdej klasy wzgledem baseline
display(per_class.round(3))

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ["mAP50-95", "F1", "train_min", "train_kWh"]):
    sns.barplot(x=df.index, y=df[col], ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(OUT / "comparison.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(per_class, annot=True, fmt=".3f", cmap="Greens", ax=axes[0])
axes[0].set_title("mAP50-95 per klasa (test)")
sns.heatmap(per_class_change, annot=True, fmt="+.3f", cmap="RdYlGn", center=0, ax=axes[1])
axes[1].set_title("zmiana wzgledem baseline")
plt.tight_layout()
plt.savefig(OUT / "per_class.png", dpi=150)
plt.show()